# Po Valley Sentinel-1 SLC: exact AOI burst manifest

**Primary output:** `metadata/s1_aoi_bursts_exact_intersection.csv` — only Sentinel-1 bursts whose footprints have positive-area intersection with the saved Po Valley AOI.

The notebook directly loads the existing `aoi/Po_Plain_distretto_intersection.gpkg` (layer `Po_Plain_distretto_intersection`). It does not rebuild the AOI or require the source boundary files. It queries the CDSE Bursts catalogue, filters the returned burst footprints against that AOI in memory, and writes only the exact-intersection manifest. It does **not** create `metadata/s1_aoi_bursts.csv` or download SAR imagery during catalogue search.


## 1. Project paths and run settings

1. Load the existing `aoi/Po_Plain_distretto_intersection.gpkg`
2. Query the CDSE Bursts catalogue for the configured date window (or selected parent SAFE products).
3. Filter the catalogue results against the exact AOI geometry in memory.

The AOI is loaded from the existing GeoPackage; no rebuild is performed. Set `REFRESH_CATALOGUE = True` to rerun the catalogue query even if the final manifest already exists; otherwise, an existing exact manifest is reused.


In [1]:
from pathlib import Path
import os, re, json, time, subprocess, shutil
import pandas as pd
from IPython.display import display

BASE_DIR = Path("/mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley_ssm")
EFFECTIVE_AOI_FILE = BASE_DIR / "aoi" / "Po_Plain_distretto_intersection.gpkg"
EFFECTIVE_AOI_LAYER = "Po_Plain_distretto_intersection"
# Exact burst-footprint intersection with the effective Po Valley AOI.
EXACT_BURST_MANIFEST = BASE_DIR / "metadata" / "s1_aoi_bursts_exact_intersection.csv"
# Reuse the exact output if present. Set True to rerun the CDSE catalogue query.
REFRESH_CATALOGUE = False
BURST_RAW_JSONL = BASE_DIR / "metadata" / "s1_aoi_burst_raw.jsonl"
BURST_QUERY_SUMMARY = BASE_DIR / "metadata" / "s1_aoi_burst_query_summary.csv"

# Prefer the redundancy-optimized SAFE plan if it exists; otherwise use the AOI product list.
OPTIMIZED_PARENTS = BASE_DIR / "metadata" / "s1_safe_download_plan_optimized.csv"
PARENT_PRODUCTS = BASE_DIR / "metadata" / "s1_aoi_products.csv"

# Fallback catalogue-search window when no precomputed parent-product list exists.
PRODUCT_QUERY_START = "2021-04-01"
PRODUCT_QUERY_END_EXCLUSIVE = "2021-11-01"  # includes acquisitions through 2021-10-31

OUTPUT_DIR = BASE_DIR / "data" / "raw" / "s1_bursts_s3"
PLAN_FILE = BASE_DIR / "metadata" / "s1_aoi_s3_extraction_plan.csv"
LOG_FILE = BASE_DIR / "metadata" / "s1_aoi_s3_extraction_log.csv"
LOCAL_MANIFEST = BASE_DIR / "metadata" / "s1_aoi_s3_local_manifest.csv"

ENV_FILE = BASE_DIR / ".env"
DOCKER_IMAGE = "cdse_utilities"

# Safety defaults: first test only one date and two extraction jobs.
PILOT_DATE = None #"2021-04-01"  # set None for all dates after the pilot succeeds
#MAX_EXTRACTIONS = 2        # set None for all planned jobs; 0 means plan only
#RUN_EXTRACTION = True     # change to True only after reviewing the plan
#BUILD_DOCKER_IMAGE = False # set True once if the image is not already built

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project:", BASE_DIR)
print("Existing AOI GeoPackage:", EFFECTIVE_AOI_FILE)
print("AOI GeoPackage exists:", EFFECTIVE_AOI_FILE.exists())
print("AOI layer:", EFFECTIVE_AOI_LAYER)
print("Exact AOI-intersection manifest:", EXACT_BURST_MANIFEST)
print("Refresh CDSE catalogue query:", REFRESH_CATALOGUE)
print("Extraction plan output:", PLAN_FILE)
print("Parent list:", OPTIMIZED_PARENTS if OPTIMIZED_PARENTS.exists() else PARENT_PRODUCTS)
print("Output directory:", OUTPUT_DIR)

Project: /mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley_ssm
Existing AOI GeoPackage: /mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley_ssm/aoi/Po_Plain_distretto_intersection.gpkg
AOI GeoPackage exists: True
AOI layer: Po_Plain_distretto_intersection
Exact AOI-intersection manifest: /mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley_ssm/metadata/s1_aoi_bursts_exact_intersection.csv
Refresh CDSE catalogue query: False
Extraction plan output: /mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley_ssm/metadata/s1_aoi_s3_extraction_plan.csv
Parent list: /mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley_ssm/metadata/s1_aoi_products.csv
Output directory: /mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley_ssm/data/raw/s1_bursts_s3


## 2. Query CDSE and write the exact AOI-intersection manifest

The catalogue is queried using simple rectangular tiles for efficiency. Results are deduplicated in memory, then each burst footprint is tested against the existing AOI geometry. Only positive-area intersections are saved to `metadata/s1_aoi_bursts_exact_intersection.csv`; no broad candidate CSV is written.

If the exact output already exists and `REFRESH_CATALOGUE = False`, the query is skipped and the existing final manifest is reused.


In [2]:
import json, time
import geopandas as gpd
import requests
from shapely.geometry import box, shape
from shapely.prepared import prep

BURSTS_ENDPOINT = "https://catalogue.dataspace.copernicus.eu/odata/v1/Bursts"
TOP = 1000
TIMEOUT_SECONDS = 120
MAX_RETRIES = 5
RETRY_BACKOFF_SECONDS = 3
MAX_AOI_WKT_CHARS = 5500
AOI_TILE_DEGREES = 0.5


def find_col(df, candidates, required=True, label="column"):
    norm = {str(c).strip().lower().replace(" ", "_").replace("-", "_"): c for c in df.columns}
    for candidate in candidates:
        key = candidate.strip().lower().replace(" ", "_").replace("-", "_")
        if key in norm:
            return norm[key]
    if required:
        raise KeyError(f"Could not identify {label}; tried {candidates}; columns={list(df.columns)}")
    return None


def parse_product_start(name):
    m = re.search(r"_(\d{8}T\d{6})_", str(name))
    if not m:
        return pd.NaT
    return pd.to_datetime(m.group(1), format="%Y%m%dT%H%M%S", utc=True, errors="coerce")


def load_selected_parents():
    """Load an optional parent list; return None when no precomputed list exists."""
    parent_file = OPTIMIZED_PARENTS if OPTIMIZED_PARENTS.exists() else PARENT_PRODUCTS
    if not parent_file.exists():
        print("No precomputed parent list found; will derive parent products from the AOI burst query.")
        return None, None
    df = pd.read_csv(parent_file, low_memory=False)
    name_col = find_col(df, ["product_name", "name", "ParentProductName", "_parent_name"], label="parent product name")
    id_col = find_col(df, ["product_id", "id", "uuid", "ParentProductId", "_parent_id"], required=False, label="parent product ID")
    out = pd.DataFrame({"parent_product_name": df[name_col].astype(str).str.strip()})
    out["parent_product_id"] = df[id_col].astype(str).str.strip() if id_col else ""
    out = out[out["parent_product_name"].ne("") & out["parent_product_name"].ne("nan")]
    out = out.drop_duplicates(["parent_product_name", "parent_product_id"]).reset_index(drop=True)
    out["_start"] = out["parent_product_name"].map(parse_product_start)
    if out["_start"].notna().sum() == 0:
        raise ValueError("Could not parse acquisition dates from parent product names.")
    return out, parent_file


def load_effective_aoi():
    """Load the already-created final AOI GeoPackage; never rebuild it here."""
    if not EFFECTIVE_AOI_FILE.exists():
        raise FileNotFoundError(
            f"Final AOI GeoPackage not found: {EFFECTIVE_AOI_FILE}. "
            "Create/copy this intersection GeoPackage before running the notebook."
        )
    aoi = gpd.read_file(EFFECTIVE_AOI_FILE, layer=EFFECTIVE_AOI_LAYER)
    if aoi.empty:
        raise ValueError(f"Final AOI layer is empty: {EFFECTIVE_AOI_FILE} [{EFFECTIVE_AOI_LAYER}]")
    if aoi.crs is None:
        raise ValueError(f"Final AOI layer has no CRS: {EFFECTIVE_AOI_FILE}")
    aoi = aoi.loc[~aoi.geometry.isna() & ~aoi.geometry.is_empty].copy()
    if aoi.empty:
        raise ValueError(f"Final AOI layer has no non-empty geometries: {EFFECTIVE_AOI_FILE}")
    aoi["geometry"] = aoi.geometry.make_valid()
    print("Loaded final AOI GeoPackage:", EFFECTIVE_AOI_FILE)
    print("AOI layer:", EFFECTIVE_AOI_LAYER)
    print("AOI CRS:", aoi.crs)
    area_km2 = aoi.to_crs("EPSG:3035").geometry.union_all().area / 1_000_000
    print("AOI area (km², measured in EPSG:3035):", round(area_km2, 3))
    return aoi
def load_aoi_wkt():
    """Create a small set of simple rectangle queries covering the effective AOI.

    The old approach intersected every tile with the highly detailed polygon and
    recursively split the resulting boundary geometry. That produced thousands
    of queries. Instead, query simple bounding rectangles for tiles that touch
    the AOI; deduplicate the returned catalogue records afterward. This is a
    candidate search, so rectangles may return bursts outside the exact AOI.
    """
    if AOI_TILE_DEGREES <= 0:
        raise ValueError("AOI_TILE_DEGREES must be positive.")
    aoi = load_effective_aoi()
    geom = aoi.to_crs("EPSG:4326").geometry.union_all()
    minx, miny, maxx, maxy = geom.bounds
    parts = []
    x = minx
    while x < maxx:
        x2 = min(x + AOI_TILE_DEGREES, maxx)
        y = miny
        while y < maxy:
            y2 = min(y + AOI_TILE_DEGREES, maxy)
            tile = box(x, y, x2, y2)
            # Only query tiles that touch the true AOI; send the rectangle, not
            # the detailed clipped polygon, to keep each OData filter tiny.
            if geom.intersects(tile):
                parts.append(tile.wkt)
            y = y2
        x = x2
    if not parts:
        raise ValueError("No query tiles intersect the effective AOI.")
    print(f"AOI bounds: ({minx:.4f}, {miny:.4f}, {maxx:.4f}, {maxy:.4f})")
    print(f"Simple candidate-query tiles: {len(parts):,} (tile size {AOI_TILE_DEGREES:g}°)")
    return parts, f"bounding_rectangle_candidate_tiles_{AOI_TILE_DEGREES:g}_degrees_{len(parts)}_parts"

def month_windows(start, end):
    cursor = start
    while cursor < end:
        next_month = (cursor + pd.offsets.MonthBegin(1)).normalize()
        stop = min(next_month, end)
        if stop <= cursor:
            stop = min(cursor + pd.Timedelta(days=31), end)
        yield cursor, stop
        cursor = stop


def iso_z(ts):
    return pd.Timestamp(ts).strftime("%Y-%m-%dT%H:%M:%S.000Z")


def get_json(session, url, params=None):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(url, params=params, timeout=TIMEOUT_SECONDS)
            if response.status_code == 429 or response.status_code >= 500:
                raise requests.HTTPError(f"HTTP {response.status_code}: {response.text[:300]}")
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            if attempt == MAX_RETRIES:
                break
            time.sleep(RETRY_BACKOFF_SECONDS * attempt)
    raise RuntimeError(f"CDSE Bursts query failed after {MAX_RETRIES} attempts: {last_error}")


def query_window(session, aoi_wkt, start, end):
    spatial = f"OData.CSC.Intersects(area=geography'SRID=4326;{aoi_wkt}')"
    date_filter = f"ContentDate/Start ge {iso_z(start)} and ContentDate/Start lt {iso_z(end)}"
    product_filter = "ParentProductType eq 'IW_SLC__1S'"
    params = {"$filter": f"{spatial} and {date_filter} and {product_filter}", "$orderby": "ContentDate/Start asc", "$top": TOP}
    rows, url, first = [], BURSTS_ENDPOINT, True
    while url:
        payload = get_json(session, url, params=params if first else None)
        first = False
        rows.extend(payload.get("value", []))
        url = payload.get("@odata.nextLink")
    return rows


def flatten_record(record):
    out = {}
    for key, value in record.items():
        if isinstance(value, dict):
            for child_key, child_value in value.items():
                out[f"{key}_{child_key}"] = json.dumps(child_value) if isinstance(child_value, (dict, list)) else child_value
        elif isinstance(value, list):
            out[key] = json.dumps(value)
        else:
            out[key] = value
    return out


if REFRESH_CATALOGUE or not EXACT_BURST_MANIFEST.exists():
    print("Running CDSE catalogue query because refresh was requested or exact manifest is missing.")
    parents, PARENT_SOURCE = load_selected_parents()
    aoi_wkts, AOI_QUERY_METHOD = load_aoi_wkt()
    if parents is not None:
        start = parents["_start"].dropna().min().floor("D")
        end = parents["_start"].dropna().max().floor("D") + pd.Timedelta(days=1)
        print(f"Parent source: {PARENT_SOURCE}")
        print(f"Selected parent products: {len(parents):,}")
    else:
        start = pd.Timestamp(PRODUCT_QUERY_START, tz="UTC")
        end = pd.Timestamp(PRODUCT_QUERY_END_EXCLUSIVE, tz="UTC")
        print("Parent source: derived from AOI-intersecting burst catalogue results")
        print(f"Configured catalogue window: {start} to {end} (end exclusive)")
    print(f"Query window: {start} to {end} (end exclusive)")
    print(f"AOI query geometry: {AOI_QUERY_METHOD}")
    print(f"URL-safe AOI query parts: {len(aoi_wkts):,}")
    print("Querying CDSE catalogue only; no SAR imagery is downloaded in this step.")

    session = requests.Session()
    session.headers.update({"User-Agent": "po-valley-s1-aoi-burst-notebook/2.3"})
    all_records = []
    BURST_RAW_JSONL.parent.mkdir(parents=True, exist_ok=True)
    windows = list(month_windows(start, end))
    total_queries = len(windows) * len(aoi_wkts)
    query_i = 0
    seen_raw_keys = set()
    with BURST_RAW_JSONL.open("w", encoding="utf-8") as raw:
        for win_start, win_end in windows:
            month_count = 0
            for part_i, aoi_wkt_part in enumerate(aoi_wkts, start=1):
                query_i += 1
                records = query_window(session, aoi_wkt_part, win_start, win_end)
                # Rectangle tiles may return the same burst more than once.
                # Deduplicate immediately to reduce memory use and output size.
                for record in records:
                    rid = record.get("Id", record.get("id"))
                    key = str(rid) if rid not in (None, "") else json.dumps(record, sort_keys=True, ensure_ascii=False)
                    if key in seen_raw_keys:
                        continue
                    seen_raw_keys.add(key)
                    raw.write(json.dumps(record, ensure_ascii=False) + "\n")
                    all_records.append(record)
                    month_count += 1
                if query_i % 10 == 0 or query_i == total_queries:
                    print(f"{query_i:,}/{total_queries:,} queries; {len(all_records):,} unique raw rows")
            print(f"{win_start.date()} → {win_end.date()}: {month_count:,} unique raw burst rows")

    if not all_records:
        raise RuntimeError("CDSE returned no bursts for this AOI/date range. Check the effective AOI, configured dates, and API response before proceeding.")

    bursts_all = pd.DataFrame([flatten_record(r) for r in all_records])
    parent_name_col_api = find_col(bursts_all, ["ParentProductName", "parent_product_name"], required=False)
    parent_id_col_api = find_col(bursts_all, ["ParentProductId", "parent_product_id"], required=False)
    if parent_name_col_api is None and parent_id_col_api is None:
        raise RuntimeError(f"CDSE response has no parent product name/ID fields. Returned columns: {list(bursts_all.columns)}")

    if parents is None:
        # The AOI + date-filtered burst response defines the parent set when no list was supplied.
        matched = bursts_all.copy()
        if parent_name_col_api:
            name_values = matched[parent_name_col_api].astype(str).str.strip()
        else:
            name_values = pd.Series("", index=matched.index)
        if parent_id_col_api:
            id_values = matched[parent_id_col_api].astype(str).str.strip()
        else:
            id_values = pd.Series("", index=matched.index)
        parents = pd.DataFrame({"parent_product_name": name_values, "parent_product_id": id_values})
        parents = parents[parents["parent_product_name"].ne("") | parents["parent_product_id"].ne("")].drop_duplicates().reset_index(drop=True)
        parents["_start"] = parents["parent_product_name"].map(parse_product_start)
    else:
        name_set = set(parents["parent_product_name"])
        id_set = set(parents.loc[parents["parent_product_id"].ne(""), "parent_product_id"])
        mask = pd.Series(False, index=bursts_all.index)
        if parent_name_col_api:
            mask |= bursts_all[parent_name_col_api].astype(str).str.strip().isin(name_set)
        if parent_id_col_api and id_set:
            mask |= bursts_all[parent_id_col_api].astype(str).str.strip().isin(id_set)
        matched = bursts_all.loc[mask].copy()
        if matched.empty:
            raise RuntimeError("CDSE returned bursts, but none matched the selected parent products by name or ID. Inspect the raw JSONL and parent list.")

    # Normalize parent identity columns in the saved manifest without creating duplicate column names.
    if "parent_product_name" in matched.columns:
        matched = matched.drop(columns=["parent_product_name"])
    if "parent_product_id" in matched.columns:
        matched = matched.drop(columns=["parent_product_id"])
    matched.insert(0, "parent_product_name", matched[parent_name_col_api].astype(str).str.strip() if parent_name_col_api else "")
    matched.insert(1, "parent_product_id", matched[parent_id_col_api].astype(str).str.strip() if parent_id_col_api else "")
    burst_uuid_col = find_col(matched, ["Id", "id"], required=False)
    if burst_uuid_col:
        matched = matched.drop_duplicates(subset=[burst_uuid_col])
    else:
        matched = matched.drop_duplicates()
    # Exact-filter catalogue results in memory; do not write a broad candidate CSV.
    aoi_gdf = load_effective_aoi()  # Load existing Po_Plain_distretto_intersection.gpkg
    if aoi_gdf.empty or aoi_gdf.crs is None:
        raise ValueError("Effective AOI layer is empty or has no CRS.")
    aoi_gdf = aoi_gdf.loc[~aoi_gdf.geometry.isna() & ~aoi_gdf.geometry.is_empty].copy()
    if aoi_gdf.empty:
        raise ValueError("Effective AOI has no non-empty geometry.")
    aoi_geom = aoi_gdf.to_crs("EPSG:4326").geometry.make_valid().union_all()
    prepared_aoi = prep(aoi_geom)
    aoi_metric = gpd.GeoSeries([aoi_geom], crs="EPSG:4326").to_crs("EPSG:3035").iloc[0]
    coords_col = find_col(matched, ["GeoFootprint_coordinates"], label="burst footprint coordinates")
    type_col = find_col(matched, ["GeoFootprint_type"], required=False, label="burst footprint geometry type")
    parsed_geometries, invalid_rows = {}, []
    for idx, row in matched.iterrows():
        raw_coords = row.get(coords_col)
        if pd.isna(raw_coords) or not str(raw_coords).strip():
            invalid_rows.append(idx)
            continue
        try:
            coords = json.loads(raw_coords) if isinstance(raw_coords, str) else raw_coords
            geom_type = row.get(type_col, "Polygon") if type_col else "Polygon"
            if pd.isna(geom_type) or not str(geom_type).strip():
                geom_type = "Polygon"
            geom = shape({"type": str(geom_type), "coordinates": coords})
            if not geom.is_valid:
                geom = geom.buffer(0)
            if geom.is_empty:
                invalid_rows.append(idx)
                continue
            parsed_geometries[idx] = geom
        except Exception:
            invalid_rows.append(idx)

    intersecting_indices = [idx for idx, geom in parsed_geometries.items() if prepared_aoi.intersects(geom)]
    positive_area_indices = []
    if intersecting_indices:
        burst_metric = gpd.GeoSeries([parsed_geometries[idx] for idx in intersecting_indices], crs="EPSG:4326").to_crs("EPSG:3035")
        for idx, geom_metric in zip(intersecting_indices, burst_metric):
            if geom_metric.intersection(aoi_metric).area > 0:
                positive_area_indices.append(idx)

    candidates_count = len(matched)
    exact = matched.loc[positive_area_indices].copy()
    if exact.empty:
        raise RuntimeError("Exact AOI intersection retained zero bursts. Check AOI CRS and catalogue footprint fields.")
    burst_uuid_col = find_col(exact, ["Id", "id"], required=False, label="catalogue burst UUID")
    exact = exact.drop_duplicates(subset=[burst_uuid_col]) if burst_uuid_col else exact.drop_duplicates()
    EXACT_BURST_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    exact.to_csv(EXACT_BURST_MANIFEST, index=False)
    matched = exact
    print(f"Catalogue rows after parent matching: {candidates_count:,}")
    print(f"Invalid/missing footprint rows: {len(set(invalid_rows)):,}")
    print(f"Exact positive-area AOI intersections saved: {len(exact):,}")
    print("Exact manifest:", EXACT_BURST_MANIFEST)

    found_names = set(matched["parent_product_name"].dropna().astype(str)) - {"", "nan", "None"}
    found_ids = set(matched["parent_product_id"].dropna().astype(str)) - {"", "nan", "None"}
    if PARENT_SOURCE is None:
        parent_has = pd.Series(True, index=parents.index)
        missing = pd.DataFrame(columns=["parent_product_name", "parent_product_id"])
    else:
        parent_has = parents["parent_product_name"].isin(found_names) | parents["parent_product_id"].isin(found_ids)
        missing = parents.loc[~parent_has, ["parent_product_name", "parent_product_id"]].copy()
    MISSING_PARENTS_FILE = BASE_DIR / "metadata" / "s1_aoi_parents_without_bursts.csv"
    missing.to_csv(MISSING_PARENTS_FILE, index=False)

    summary = pd.DataFrame([{
        "parent_source": str(PARENT_SOURCE) if PARENT_SOURCE is not None else "derived_from_AOI_burst_query", "parent_products": len(parents),
        "catalogue_burst_rows_returned_including_tile_duplicates": len(bursts_all), "exact_intersection_unique_bursts": len(matched),
        "parents_with_bursts": int(parent_has.sum()), "parents_without_bursts": int((~parent_has).sum()),
        "query_start_utc": start.isoformat(), "query_end_exclusive_utc": end.isoformat(),
        "aoi_query_geometry_method": AOI_QUERY_METHOD, "burst_endpoint": BURSTS_ENDPOINT,
    }])
    summary.to_csv(BURST_QUERY_SUMMARY, index=False)
    print("\nGenerated exact-intersection manifest:", EXACT_BURST_MANIFEST)
    print("Raw API audit:", BURST_RAW_JSONL)
    print("Query summary:", BURST_QUERY_SUMMARY)
    print("Parents without matched bursts:", MISSING_PARENTS_FILE)
    print(f"Exact-intersection catalogue bursts: {len(matched):,}")
    print(f"Parents with bursts: {int(parent_has.sum()):,} / {len(parents):,}")
    display(matched.head(5))

else:
    print("Reusing existing exact-intersection manifest; no CDSE query will run:", EXACT_BURST_MANIFEST)
    exact_check = pd.read_csv(EXACT_BURST_MANIFEST, low_memory=False)
    if exact_check.empty:
        raise RuntimeError(f"Existing exact-intersection manifest is empty: {EXACT_BURST_MANIFEST}")
    print(f"Loaded exact-intersection rows: {len(exact_check):,}")
    print(f"Unique parent products: {exact_check.get('parent_product_name', pd.Series(dtype=str)).nunique():,}")


Running CDSE catalogue query because refresh was requested or exact manifest is missing.
No precomputed parent list found; will derive parent products from the AOI burst query.
Loaded final AOI GeoPackage: /mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley_ssm/aoi/Po_Plain_distretto_intersection.gpkg
AOI layer: Po_Plain_distretto_intersection
AOI CRS: EPSG:4326
AOI area (km², measured in EPSG:3035): 38977.49
AOI bounds: (7.2724, 43.9172, 12.8037, 45.8061)
Simple candidate-query tiles: 37 (tile size 0.5°)
Parent source: derived from AOI-intersecting burst catalogue results
Configured catalogue window: 2021-04-01 00:00:00+00:00 to 2021-11-01 00:00:00+00:00 (end exclusive)
Query window: 2021-04-01 00:00:00+00:00 to 2021-11-01 00:00:00+00:00 (end exclusive)
AOI query geometry: bounding_rectangle_candidate_tiles_0.5_degrees_37_parts
URL-safe AOI query parts: 37
Querying CDSE catalogue only; no SAR imagery is downloaded in this step.
10/259 queries; 1,066 unique raw rows
20/259 queries; 1,516 un

,parent_product_name,parent_product_id,@odata.mediaContentType,Id,Name,ContentType,S3Path,ContentDate_Start,ContentDate_End,Footprint,...,OrbitDirection,PlatformSerialIdentifier,PolarisationChannels,OperationalMode,DatatakeID,Lines,LinesPerBurst,SamplesPerBurst,BeginningDateTime,EndingDateTime
10,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,551588e6-97db-53c1-90f4-65172cea2d49,application/octet-stream,7b5a9850-4b6e-47aa-b4ae-96dca4620e04,S1A-SLC-20210401T172310-187130-IW2-VV-287642,application/octet-stream,/eodata/Sentinel-1/SAR/SLC/2021/04/01/S1A_IW_S...,2021-04-01T17:23:10.246472Z,2021-04-01T17:23:11.287372Z,geography'SRID=4326;POLYGON ((7.244755 44.1087...,...,ASCENDING,A,VV,IW,287642,0,1511,26345,2021-04-01T17:23:10.246472Z,2021-04-01T17:23:11.287372Z
11,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,551588e6-97db-53c1-90f4-65172cea2d49,application/octet-stream,b5a3fc64-2aab-4198-ba70-56d4c50d8494,S1A-SLC-20210401T172310-187130-IW2-VH-287642,application/octet-stream,/eodata/Sentinel-1/SAR/SLC/2021/04/01/S1A_IW_S...,2021-04-01T17:23:10.246472Z,2021-04-01T17:23:11.287372Z,geography'SRID=4326;POLYGON ((7.244755 44.1087...,...,ASCENDING,A,VH,IW,287642,0,1511,26345,2021-04-01T17:23:10.246472Z,2021-04-01T17:23:11.287372Z
12,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,551588e6-97db-53c1-90f4-65172cea2d49,application/octet-stream,58c8f1a8-6177-4086-8e35-638ea287fdd0,S1A-SLC-20210401T172313-187131-IW2-VV-287642,application/octet-stream,/eodata/Sentinel-1/SAR/SLC/2021/04/01/S1A_IW_S...,2021-04-01T17:23:13.004749Z,2021-04-01T17:23:14.045649Z,geography'SRID=4326;POLYGON ((7.206643 44.2747...,...,ASCENDING,A,VV,IW,287642,1511,1511,26345,2021-04-01T17:23:13.004749Z,2021-04-01T17:23:14.045649Z
13,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,551588e6-97db-53c1-90f4-65172cea2d49,application/octet-stream,59f49ee2-3909-404a-809b-73c5bac1a530,S1A-SLC-20210401T172313-187131-IW2-VH-287642,application/octet-stream,/eodata/Sentinel-1/SAR/SLC/2021/04/01/S1A_IW_S...,2021-04-01T17:23:13.004749Z,2021-04-01T17:23:14.045649Z,geography'SRID=4326;POLYGON ((7.206643 44.2747...,...,ASCENDING,A,VH,IW,287642,1511,1511,26345,2021-04-01T17:23:13.004749Z,2021-04-01T17:23:14.045649Z
14,S1A_IW_SLC__1SDV_20210405T054415_20210405T0544...,71271315-625a-51c8-9e0d-4df05698224b,application/octet-stream,0c86ebfc-76ab-4d5a-862e-da5539df8f8b,S1A-SLC-20210405T054418-297224-IW1-VV-288089,application/octet-stream,/eodata/Sentinel-1/SAR/SLC/2021/04/05/S1A_IW_S...,2021-04-05T05:44:18.814681Z,2021-04-05T05:44:19.692308Z,geography'SRID=4326;POLYGON ((7.601734 44.4559...,...,DESCENDING,A,VV,IW,288089,0,1507,22794,2021-04-05T05:44:18.814681Z,2021-04-05T05:44:19.692308Z


## 3. Validate the exact manifest

The catalogue query cell has already applied the exact positive-area AOI filter and written the final manifest. This cell only validates that output; it does not read or require an intermediate candidate CSV.


In [3]:
if not EXACT_BURST_MANIFEST.exists():
    raise FileNotFoundError(
        f"Exact burst manifest was not generated: {EXACT_BURST_MANIFEST}. Run the catalogue query cell first."
    )
exact_manifest_check = pd.read_csv(EXACT_BURST_MANIFEST, low_memory=False)
if exact_manifest_check.empty:
    raise RuntimeError(f"Exact burst manifest is empty: {EXACT_BURST_MANIFEST}")
print("Exact AOI-intersection manifest validated:", EXACT_BURST_MANIFEST)
print(f"Rows: {len(exact_manifest_check):,}")
print("Unique parent products:", exact_manifest_check.get("parent_product_name", pd.Series(dtype=str)).nunique())
display(exact_manifest_check.head(10))


Exact AOI-intersection manifest validated: /mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley_ssm/metadata/s1_aoi_bursts_exact_intersection.csv
Rows: 11,018
Unique parent products: 599


,parent_product_name,parent_product_id,@odata.mediaContentType,Id,Name,ContentType,S3Path,ContentDate_Start,ContentDate_End,Footprint,...,OrbitDirection,PlatformSerialIdentifier,PolarisationChannels,OperationalMode,DatatakeID,Lines,LinesPerBurst,SamplesPerBurst,BeginningDateTime,EndingDateTime
0,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,551588e6-97db-53c1-90f4-65172cea2d49,application/octet-stream,7b5a9850-4b6e-47aa-b4ae-96dca4620e04,S1A-SLC-20210401T172310-187130-IW2-VV-287642,application/octet-stream,/eodata/Sentinel-1/SAR/SLC/2021/04/01/S1A_IW_S...,2021-04-01T17:23:10.246472Z,2021-04-01T17:23:11.287372Z,geography'SRID=4326;POLYGON ((7.244755 44.1087...,...,ASCENDING,A,VV,IW,287642,0,1511,26345,2021-04-01T17:23:10.246472Z,2021-04-01T17:23:11.287372Z
1,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,551588e6-97db-53c1-90f4-65172cea2d49,application/octet-stream,b5a3fc64-2aab-4198-ba70-56d4c50d8494,S1A-SLC-20210401T172310-187130-IW2-VH-287642,application/octet-stream,/eodata/Sentinel-1/SAR/SLC/2021/04/01/S1A_IW_S...,2021-04-01T17:23:10.246472Z,2021-04-01T17:23:11.287372Z,geography'SRID=4326;POLYGON ((7.244755 44.1087...,...,ASCENDING,A,VH,IW,287642,0,1511,26345,2021-04-01T17:23:10.246472Z,2021-04-01T17:23:11.287372Z
2,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,551588e6-97db-53c1-90f4-65172cea2d49,application/octet-stream,58c8f1a8-6177-4086-8e35-638ea287fdd0,S1A-SLC-20210401T172313-187131-IW2-VV-287642,application/octet-stream,/eodata/Sentinel-1/SAR/SLC/2021/04/01/S1A_IW_S...,2021-04-01T17:23:13.004749Z,2021-04-01T17:23:14.045649Z,geography'SRID=4326;POLYGON ((7.206643 44.2747...,...,ASCENDING,A,VV,IW,287642,1511,1511,26345,2021-04-01T17:23:13.004749Z,2021-04-01T17:23:14.045649Z
3,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,551588e6-97db-53c1-90f4-65172cea2d49,application/octet-stream,59f49ee2-3909-404a-809b-73c5bac1a530,S1A-SLC-20210401T172313-187131-IW2-VH-287642,application/octet-stream,/eodata/Sentinel-1/SAR/SLC/2021/04/01/S1A_IW_S...,2021-04-01T17:23:13.004749Z,2021-04-01T17:23:14.045649Z,geography'SRID=4326;POLYGON ((7.206643 44.2747...,...,ASCENDING,A,VH,IW,287642,1511,1511,26345,2021-04-01T17:23:13.004749Z,2021-04-01T17:23:14.045649Z
4,S1A_IW_SLC__1SDV_20210405T054415_20210405T0544...,71271315-625a-51c8-9e0d-4df05698224b,application/octet-stream,0c86ebfc-76ab-4d5a-862e-da5539df8f8b,S1A-SLC-20210405T054418-297224-IW1-VV-288089,application/octet-stream,/eodata/Sentinel-1/SAR/SLC/2021/04/05/S1A_IW_S...,2021-04-05T05:44:18.814681Z,2021-04-05T05:44:19.692308Z,geography'SRID=4326;POLYGON ((7.601734 44.4559...,...,DESCENDING,A,VV,IW,288089,0,1507,22794,2021-04-05T05:44:18.814681Z,2021-04-05T05:44:19.692308Z
5,S1A_IW_SLC__1SDV_20210405T054415_20210405T0544...,71271315-625a-51c8-9e0d-4df05698224b,application/octet-stream,9abb2579-f75b-4b6f-a704-7de447b73949,S1A-SLC-20210405T054418-297224-IW1-VH-288089,application/octet-stream,/eodata/Sentinel-1/SAR/SLC/2021/04/05/S1A_IW_S...,2021-04-05T05:44:18.814681Z,2021-04-05T05:44:19.692308Z,geography'SRID=4326;POLYGON ((7.601734 44.4559...,...,DESCENDING,A,VH,IW,288089,0,1507,22794,2021-04-05T05:44:18.814681Z,2021-04-05T05:44:19.692308Z
6,S1B_IW_SLC__1SDV_20210406T053520_20210406T0535...,2dddfd3d-8bdd-54bf-8521-1cfa45dada3a,application/octet-stream,21a6051e-263b-4251-848b-e86ea75dd242,S1A-SLC-20210406T053526-140426-IW2-VV-206076,application/octet-stream,/eodata/Sentinel-1/SAR/SLC/2021/04/06/S1B_IW_S...,2021-04-06T05:35:26.143206Z,2021-04-06T05:35:27.186173Z,geography'SRID=4326;POLYGON ((8.573726 44.4166...,...,DESCENDING,B,VV,IW,206076,1514,1514,25594,2021-04-06T05:35:26.143206Z,2021-04-06T05:35:27.186173Z
7,S1B_IW_SLC__1SDV_20210406T053520_20210406T0535...,2dddfd3d-8bdd-54bf-8521-1cfa45dada3a,application/octet-stream,9bbba998-06d0-4e59-9b00-1c40452e8c17,S1A-SLC-20210406T053526-140426-IW2-VH-206076,application/octet-stream,/eodata/Sentinel-1/SAR/SLC/2021/04/06/S1B_IW_S...,2021-04-06T05:35:26.143206Z,2021-04-06T05:35:27.186173Z,geography'SRID=4326;POLYGON ((8.573726 44

## 4. Map the exact filtered manifest columns

The previous cell generates the burst CSV. This cell reloads that CSV and checks that it includes parent product, subswath, polarization, and the relative burst ID required by the extractor.


In [4]:
if not EXACT_BURST_MANIFEST.exists():
    raise FileNotFoundError(f"Exact AOI-intersection manifest was not generated: {EXACT_BURST_MANIFEST}")
bursts = pd.read_csv(EXACT_BURST_MANIFEST, low_memory=False)
print(f"Loaded EXACT AOI-intersection manifest: {len(bursts):,} rows from {EXACT_BURST_MANIFEST}")
if len(bursts) >= 17212:
    raise RuntimeError("Safety stop: downstream manifest has >=17,212 rows, matching the broad candidate count. Re-run the exact intersection cell and inspect its summary before making an extraction plan.")

def find_col(df, candidates, required=True, label="column"):
    norm = {str(c).strip().lower().replace(" ", "_"): c for c in df.columns}
    for candidate in candidates:
        key = candidate.strip().lower().replace(" ", "_")
        if key in norm:
            return norm[key]
    if required:
        raise KeyError(f"Could not identify {label}. Tried {candidates}. Available: {list(df.columns)}")
    return None

COL_PARENT_NAME = find_col(bursts, ["parent_product_name", "ParentProductName", "product_name", "name"], label="parent SAFE product name")
COL_PARENT_ID = find_col(bursts, ["parent_product_id", "ParentProductId", "product_id", "id"], required=False, label="parent product ID")
COL_SWATH = find_col(bursts, ["SwathIdentifier", "swath", "subswath"], label="subswath")
COL_POL = find_col(bursts, ["PolarisationChannels", "PolarizationChannels", "polarisation", "polarization", "pol"], label="polarization")
COL_BURST = find_col(bursts, ["BurstId", "burst_id", "relative_burst_id"], label="relative burst ID")
COL_DATE = find_col(bursts, ["ContentDate_Start", "content_date_start", "start", "datetime", "date", "acq_date"], required=False, label="acquisition date")
COL_CATALOGUE_BURST_ID = find_col(bursts, ["Id", "id"], required=False, label="catalogue burst UUID")

print({
    "parent_name": COL_PARENT_NAME,
    "parent_id": COL_PARENT_ID,
    "swath": COL_SWATH,
    "polarization": COL_POL,
    "relative_burst_id": COL_BURST,
    "date": COL_DATE,
    "catalogue_burst_uuid (audit only)": COL_CATALOGUE_BURST_ID,
})

Loaded EXACT AOI-intersection manifest: 11,018 rows from /mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley_ssm/metadata/s1_aoi_bursts_exact_intersection.csv
{'parent_name': 'parent_product_name', 'parent_id': 'parent_product_id', 'swath': 'SwathIdentifier', 'polarization': 'PolarisationChannels', 'relative_burst_id': 'BurstId', 'date': 'ContentDate_Start', 'catalogue_burst_uuid (audit only)': 'Id'}


## 5. Restrict the exact manifest to the selected parent SAFE set (optional)

This is an additional restriction after exact AOI footprint filtering. The extraction plan is built directly from `s1_aoi_bursts_exact_intersection.csv`, the only burst manifest written by this notebook. A selected parent SAFE list can narrow the exact manifest further, but cannot add rows outside it.

In [5]:
if OPTIMIZED_PARENTS.exists() or PARENT_PRODUCTS.exists():
    parent_file = OPTIMIZED_PARENTS if OPTIMIZED_PARENTS.exists() else PARENT_PRODUCTS
    parents_filter = pd.read_csv(parent_file, low_memory=False)
    PARENT_NAME_COL = find_col(parents_filter, ["product_name", "name", "ParentProductName", "_parent_name"], label="parent product name in parent list")
    PARENT_ID_COL = find_col(parents_filter, ["product_id", "id", "uuid", "ParentProductId", "_parent_id"], required=False, label="parent product ID in parent list")

    parent_names = set(parents_filter[PARENT_NAME_COL].dropna().astype(str).str.strip())
    parent_ids = set()
    if PARENT_ID_COL:
        parent_ids = set(parents_filter[PARENT_ID_COL].dropna().astype(str).str.strip()) - {"", "nan", "None"}

    name_match = bursts[COL_PARENT_NAME].astype(str).str.strip().isin(parent_names)
    if COL_PARENT_ID and parent_ids:
        id_match = bursts[COL_PARENT_ID].astype(str).str.strip().isin(parent_ids)
        keep_mask = name_match | id_match
    else:
        keep_mask = name_match

    filtered = bursts.loc[keep_mask].copy()
    print(f"Parent products in chosen parent list: {len(parents_filter):,}")
    print(f"Burst manifest rows: {len(bursts):,}")
    print(f"Rows matched to selected parents: {len(filtered):,}")
    print(f"Rows not matched: {len(bursts) - len(filtered):,}")
    if filtered.empty:
        raise RuntimeError("No burst rows matched the chosen parent list. Check parent name/ID columns before proceeding.")
else:
    # The burst manifest was queried directly from the effective AOI and configured date range.
    filtered = bursts.copy()
    print("No precomputed parent list supplied; using all AOI/date-filtered burst rows.")
    print(f"Burst manifest rows used: {len(filtered):,}")


No precomputed parent list supplied; using all AOI/date-filtered burst rows.
Burst manifest rows used: 11,018


## 6. Create one extraction job per product × subswath × polarization × relative burst

The catalogue manifest may contain multiple representations of the same extraction request. This cell normalizes common `IW1`/`iw1` and `VV VH`/`VV,VH` formats and creates a de-duplicated job table. We preserve the catalogue burst UUID separately for traceability; the extractor's `-r` argument receives the relative burst ID.

If a polarization field is missing or the burst ID is not numeric, inspect the manifest rather than guessing.

In [6]:
def norm_swath(value):
    s = str(value).strip().lower().replace(" ", "")
    if s in {"1", "iw1"}: return "iw1"
    if s in {"2", "iw2"}: return "iw2"
    if s in {"3", "iw3"}: return "iw3"
    if s in {"1", "ew1"}: return "ew1"
    return s

def split_pols(value):
    s = str(value).upper().replace("[", "").replace("]", "").replace("'", "").replace('"', "")
    found = re.findall(r"VV|VH|HH|HV", s)
    # Preserve order but remove duplicate tokens.
    return list(dict.fromkeys(found))

def safe_component(value):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value).strip()).strip("._") or "unknown"

jobs = pd.DataFrame({
    "parent_product_name": filtered[COL_PARENT_NAME].astype(str).str.strip(),
    "swath": filtered[COL_SWATH].map(norm_swath),
    "relative_burst_id": filtered[COL_BURST].astype(str).str.strip(),
})
jobs["parent_product_id"] = filtered[COL_PARENT_ID].astype(str).str.strip().values if COL_PARENT_ID else ""
jobs["catalogue_burst_uuid"] = filtered[COL_CATALOGUE_BURST_ID].astype(str).str.strip().values if COL_CATALOGUE_BURST_ID else ""
jobs["acquisition_datetime"] = filtered[COL_DATE].astype(str).values if COL_DATE else ""
jobs["polarizations"] = filtered[COL_POL].map(split_pols)
jobs = jobs.explode("polarizations").rename(columns={"polarizations": "polarization"})
jobs["polarization"] = jobs["polarization"].astype(str).str.lower()
jobs = jobs[jobs["polarization"].isin(["vv", "vh", "hh", "hv"])].copy()

# Remove duplicate extraction requests, while keeping date/product/swath/pol/burst identity.
job_key = ["parent_product_name", "swath", "relative_burst_id", "polarization"]
jobs = jobs.drop_duplicates(job_key).reset_index(drop=True)

# Explicitly validate the extractor inputs.
valid_swaths = {"iw1", "iw2", "iw3", "ew1", "ew2", "ew3", "ew4", "ew5"}
jobs["relative_burst_id_numeric"] = pd.to_numeric(jobs["relative_burst_id"], errors="coerce")
invalid = jobs[~jobs["swath"].isin(valid_swaths) | jobs["relative_burst_id_numeric"].isna()]
if not invalid.empty:
    display(invalid.head(20))
    raise ValueError("Some rows have invalid subswath or non-numeric relative burst ID. Inspect them before continuing.")
jobs["relative_burst_id"] = jobs["relative_burst_id_numeric"].astype(int).astype(str)
jobs = jobs.drop(columns="relative_burst_id_numeric")

print(f"Unique extraction jobs (after de-duplication): {len(jobs):,}")
display(jobs.head(10))

Unique extraction jobs (after de-duplication): 11,018


,parent_product_name,swath,relative_burst_id,parent_product_id,catalogue_burst_uuid,acquisition_datetime,polarization
0,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,iw2,187130,551588e6-97db-53c1-90f4-65172cea2d49,7b5a9850-4b6e-47aa-b4ae-96dca4620e04,2021-04-01T17:23:10.246472Z,vv
1,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,iw2,187130,551588e6-97db-53c1-90f4-65172cea2d49,b5a3fc64-2aab-4198-ba70-56d4c50d8494,2021-04-01T17:23:10.246472Z,vh
2,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,iw2,187131,551588e6-97db-53c1-90f4-65172cea2d49,58c8f1a8-6177-4086-8e35-638ea287fdd0,2021-04-01T17:23:13.004749Z,vv
3,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,iw2,187131,551588e6-97db-53c1-90f4-65172cea2d49,59f49ee2-3909-404a-809b-73c5bac1a530,2021-04-01T17:23:13.004749Z,vh
4,S1A_IW_SLC__1SDV_20210405T054415_20210405T0544...,iw1,297224,71271315-625a-51c8-9e0d-4df05698224b,0c86ebfc-76ab-4d5a-862e-da5539df8f8b,2021-04-05T05:44:18.814681Z,vv
5,S1A_IW_SLC__1SDV_20210405T054415_20210405T0544...,iw1,297224,71271315-625a-51c8-9e0d-4df05698224b,9abb2579-f75b-4b6f-a704-7de447b73949,2021-04-05T05:44:18.814681Z,vh
6,S1B_IW_SLC__1SDV_20210406T053520_20210406T0535...,iw2,140426,2dddfd3d-8bdd-54bf-8521-1cfa45dada3a,21a6051e-263b-4251-848b-e86ea75dd242,2021-04-06T05:35:26.143206Z,vv
7,S1B_IW_SLC__1SDV_20210406T053520_20210406T0535...,iw2,140426,2dddfd3d-8bdd-54bf-8521-1cfa45dada3a,9bbba998-06d0-4e59-9b00-1c40452e8c17,2021-04-06T05:35:26.143206Z,vh
8,S1B_IW_SLC__1SDV_20210406T053520_20210406T0535...,iw3,140426,2dddfd3d-8bdd-54bf-8521-1cfa45dada3a,05ac58d5-1fd4-40ce-9dab-5a39e0e03804,2021-04-06T05:35:27.221243Z,vv
9,S1B_IW_SLC__1SDV_20210406T053520_20210406T0535...,iw3,140426,2dddfd3d-8bdd-54bf-8521-1cfa45dada3a,b0b22df5-acfd-4068-8bd1-9f30eeed10a2,2021-04-06T05:35:27.221243Z,vh


## 7. Apply the pilot date and create the extraction plan

The date filter is applied to the extraction plan only. It does not modify the source manifest. If the date column is missing or cannot be parsed, the notebook refuses to pretend it has filtered by date.

In [7]:
run_jobs = jobs.copy()
if PILOT_DATE is not None:
    if not COL_DATE:
        raise ValueError("PILOT_DATE is set but no date column was identified in the manifest.")
    run_jobs["_parsed_date"] = pd.to_datetime(run_jobs["acquisition_datetime"], errors="coerce", utc=True).dt.strftime("%Y-%m-%d")
    n_before = len(run_jobs)
    run_jobs = run_jobs[run_jobs["_parsed_date"] == PILOT_DATE].drop(columns="_parsed_date").copy()
    print(f"Date filter {PILOT_DATE}: {n_before:,} → {len(run_jobs):,} jobs")

run_jobs["output_prefix"] = run_jobs.apply(
    lambda r: f"{safe_component(r['parent_product_name'])}__{r['swath']}__{r['polarization']}__burst-{r['relative_burst_id']}", axis=1
)
run_jobs["expected_output_dir"] = run_jobs["output_prefix"].map(lambda x: str(OUTPUT_DIR / x))
run_jobs.to_csv(PLAN_FILE, index=False)
print("Plan saved:", PLAN_FILE)
print("Unique parent products:", run_jobs["parent_product_name"].nunique())
print("Swaths:", sorted(run_jobs["swath"].unique()))
print("Polarizations:", sorted(run_jobs["polarization"].unique()))
display(run_jobs.head(20))

Plan saved: /mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley_ssm/metadata/s1_aoi_s3_extraction_plan.csv
Unique parent products: 599
Swaths: ['iw1', 'iw2', 'iw3']
Polarizations: ['vh', 'vv']


,parent_product_name,swath,relative_burst_id,parent_product_id,catalogue_burst_uuid,acquisition_datetime,polarization,output_prefix,expected_output_dir
0,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,iw2,187130,551588e6-97db-53c1-90f4-65172cea2d49,7b5a9850-4b6e-47aa-b4ae-96dca4620e04,2021-04-01T17:23:10.246472Z,vv,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,/mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley...
1,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,iw2,187130,551588e6-97db-53c1-90f4-65172cea2d49,b5a3fc64-2aab-4198-ba70-56d4c50d8494,2021-04-01T17:23:10.246472Z,vh,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,/mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley...
2,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,iw2,187131,551588e6-97db-53c1-90f4-65172cea2d49,58c8f1a8-6177-4086-8e35-638ea287fdd0,2021-04-01T17:23:13.004749Z,vv,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,/mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley...
3,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,iw2,187131,551588e6-97db-53c1-90f4-65172cea2d49,59f49ee2-3909-404a-809b-73c5bac1a530,2021-04-01T17:23:13.004749Z,vh,S1A_IW_SLC__1SDV_20210401T172307_20210401T1723...,/mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley...
4,S1A_IW_SLC__1SDV_20210405T054415_20210405T0544...,iw1,297224,71271315-625a-51c8-9e0d-4df05698224b,0c86ebfc-76ab-4d5a-862e-da5539df8f8b,2021-04-05T05:44:18.814681Z,vv,S1A_IW_SLC__1SDV_20210405T054415_20210405T0544...,/mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley...
5,S1A_IW_SLC__1SDV_20210405T054415_20210405T0544...,iw1,297224,71271315-625a-51c8-9e0d-4df05698224b,9abb2579-f75b-4b6f-a704-7de447b73949,2021-04-05T05:44:18.814681Z,vh,S1A_IW_SLC__1SDV_20210405T054415_20210405T0544...,/mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley...
6,S1B_IW_SLC__1SDV_20210406T053520_20210406T0535...,iw2,140426,2dddfd3d-8bdd-54bf-8521-1cfa45dada3a,21a6051e-263b-4251-848b-e86ea75dd242,2021-04-06T05:35:26.143206Z,vv,S1B_IW_SLC__1SDV_20210406T053520_20210406T0535...,/mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley...
7,S1B_IW_SLC__1SDV_20210406T053520_20210406T0535...,iw2,140426,2dddfd3d-8bdd-54bf-8521-1cfa45dada3a,9bbba998-06d0-4e59-9b00-1c40452e8c17,2021-04-06T05:35:26.143206Z,vh,S1B_IW_SLC__1SDV_20210406T053520_20210406T0535...,/mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley...
8,S1B_IW_SLC__1SDV_20210406T053520_20210406T0535...,iw3,140426,2dddfd3d-8bdd-54bf-8521-1cfa45dada3a,05ac58d5-1fd4-40ce-9dab-5a39e0e03804,2021-04-06T05:35:27.221243Z,vv,S1B_IW_SLC__1SDV_20210406T053520_20210406T0535...,/mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley...
9,S1B_IW_SLC__1SDV_20210406T053520_20210406T0535...,iw3,140426,2dddfd3d-8bdd-54bf-8521-1cfa45dada3a,b0b22df5-acfd-4068-8bd1-9f30eeed10a2,2021-04-06T05:35:27.221243Z,vh,S1B_IW_SLC__1SDV_20210406T053520_20210406T0535...,/mnt/CEPH_PROJECTS/ARGOS/01_abhishek/po_valley...


## 8. What this notebook does

**It does:**
- loads the existing `aoi/Po_Plain_distretto_intersection.gpkg` layer `Po_Plain_distretto_intersection` directly;
- queries the CDSE Bursts catalogue when the exact manifest is missing or `REFRESH_CATALOGUE = True`;
- filters catalogue burst footprints against the AOI in memory and writes only `metadata/s1_aoi_bursts_exact_intersection.csv`;
- optionally saves a raw JSONL API audit, query summary, and parents-without-bursts report;
- builds a de-duplicated extraction plan from the exact manifest and supports a controlled S3 burst extraction pilot.

